# 🏥 Operation: Clean Claims
## Healthcare Fraud Detection Workshop — Knowledge Graph Challenge

---

### Your mission

HealthGuard Analytics has flagged the 2024 claims dataset for potential fraud.
Your team's job: **find all fraud patterns using knowledge graph techniques**.

This is a team competition. Five fraud patterns are hidden in the data.
Each pattern you correctly identify scores **20 points** — maximum **100 points**.

> Submit your findings by calling `score_team('Team Name')` at the bottom of this notebook.

---

### Scoring

| Points | Condition |
|--------|-----------|
| 20 | Correct fraud type + correct entity (provider or patient ID) |
| +5 bonus | Supported by a graph visualisation |

---

### Dataset

| File | Description |
|------|-------------|
| `data/claims_public.csv` | ~2,000 healthcare claims — **no fraud labels** |
| `data/patients.csv` | Patient demographics |
| `data/providers.csv` | Provider metadata |
| `data/referrals.csv` | Provider-to-provider referrals |

Claim columns: `claim_id`, `patient_id`, `provider_id`, `procedure_code`, `procedure_desc`, `diagnosis_code`, `date_of_service`, `amount_billed`, `amount_paid`, `claim_status`

---

### Five fraud types to hunt for

1. **Ghost Billing** — claims filed for patients who are deceased
2. **Referral Ring** — providers that only refer to each other in a closed loop
3. **Impossible Travel** — a patient with same-day claims in geographically distant cities
4. **Upcoding** — a provider billing the most complex (expensive) procedure code at an extreme rate
5. **Duplicate Billing** — the same claim submitted multiple times

Good luck. 🕵️

## Setup — run this first

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
import community as community_louvain
from pyvis.network import Network
from collections import Counter

pd.set_option('display.max_columns', 30)
plt.style.use('seaborn-v0_8-darkgrid')
print('Ready. Good luck! 🕵️')

In [ ]:
# Use claims_public.csv — no fraud labels
patients  = pd.read_csv('../data/patients.csv',        parse_dates=['date_of_birth', 'date_of_death'])
providers = pd.read_csv('../data/providers.csv')
claims    = pd.read_csv('../data/claims_public.csv',   parse_dates=['date_of_service'])
referrals = pd.read_csv('../data/referrals.csv',       parse_dates=['referral_date'])

print(f'Patients:  {len(patients):,}')
print(f'Providers: {len(providers):,}')
print(f'Claims:    {len(claims):,}')
print(f'Referrals: {len(referrals):,}')
print(f'Total billed: ${claims["amount_billed"].sum():,.2f}')

## Explore the Data

Before building any graphs, get a feel for the dataset.
Some questions worth exploring:

- What is the distribution of `amount_billed`? Are there obvious outliers?
- Which providers have the highest average claim amount?
- Are all patients alive? (Check `date_of_death`…)
- What does the `procedure_code` distribution look like — do any providers deviate from the population?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(claims['amount_billed'], bins=60, color='#3498db', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Amount Billed ($)')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Claim Amounts')

top = (claims.groupby('provider_id')['amount_billed']
       .agg(['count','mean']).sort_values('mean', ascending=False).head(20))
axes[1].bar(range(len(top)), top['mean'], color='#e67e22', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('Top-20 providers by avg claim')
axes[1].set_ylabel('Average claim ($)')
axes[1].set_title('Top 20 Providers by Average Claim Amount')
axes[1].set_xticks(range(len(top)))
axes[1].set_xticklabels(top.index, rotation=45, ha='right', fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
# Procedure code distribution overall — useful baseline for fraud 4
print('Overall procedure code distribution:')
print(claims['procedure_code'].value_counts().to_string())

In [ ]:
# How many patients have a date_of_death?
print(f'Patients with recorded date of death: {patients["date_of_death"].notna().sum()}')
print(patients[patients['date_of_death'].notna()][['patient_id','first_name','last_name','date_of_death']])

## Build Your Knowledge Graph

Below is scaffolding for the core graphs. Add to it as needed.

The fundamental insight: **things that look normal in a flat table become anomalies in a graph**.

For example:
- A provider node with unusually high *degree* (many unique patients) may be worth investigating
- A patient node connected to providers in different cities is suspicious
- A small community in the referral graph that has no edges to the outside world is unusual

In [ ]:
# Bipartite patient-provider multigraph
# Each claim becomes an edge; multi-edges allowed (same patient can visit same provider multiple times)
G = nx.MultiGraph()

for _, row in patients.iterrows():
    G.add_node(row['patient_id'],
               ntype='patient',
               city=row['city'],
               deceased=pd.notna(row['date_of_death']))

for _, row in providers.iterrows():
    G.add_node(row['provider_id'],
               ntype='provider',
               city=row['city'],
               name=row['provider_name'])

for _, row in claims.iterrows():
    G.add_edge(row['patient_id'], row['provider_id'],
               claim_id=row['claim_id'],
               amount=row['amount_billed'],
               date=row['date_of_service'],
               procedure=row['procedure_code'])

print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

In [ ]:
# Directed provider-to-provider referral graph
R = nx.DiGraph()
for _, row in referrals.iterrows():
    u, v = row['referring_provider_id'], row['referred_provider_id']
    if R.has_edge(u, v):
        R[u][v]['weight'] += 1
    else:
        R.add_edge(u, v, weight=1)

print(f'Referral graph: {R.number_of_nodes()} nodes, {R.number_of_edges()} edges')

---
## 🔍 Challenge 1 — The Dead Don't Visit Doctors (20 pts)

**Background:** The `patients` table has a `date_of_death` column. A handful of patients have a non-null value — they passed away before 2024. Yet someone is still billing for them.

**Your task:** Find any claims where `date_of_service` falls *after* the patient's `date_of_death`.

**Graph angle:** These are temporal anomalies on the patient→provider edges. Build a subgraph of the suspicious provider and highlight deceased patient nodes.

*Hint: a DataFrame merge and date comparison is sufficient. The graph makes it visual.*

In [ ]:
# Challenge 1 — your code here

# Step 1: identify deceased patients
deceased = patients[patients['date_of_death'].notna()]
print(f'Deceased patients: {len(deceased)}')

# Step 2: join claims with deceased patients, filter for service AFTER death
# ...


In [ ]:
# Challenge 1 — visualisation (bonus +5 pts)
# Build a small subgraph: suspicious provider + the deceased patients they billed
# Hint: nx.bipartite_layout(subgraph, [provider_id]) gives a clean layout

# ...

---
## 🔍 Challenge 2 — The Inner Circle (20 pts)

**Background:** Most providers refer patients broadly across the network. But a small group might operate in a closed loop — only referring *within* themselves, never outward.

**Your task:** Use community detection on the referral graph to find a group of providers that are completely isolated from the rest of the network. Also check whether their claim amounts are unusual.

**Graph angle:** Louvain community detection on the undirected projection of `R`. A community with zero external edges is a strong fraud signal.

*Hint: `community_louvain.best_partition(R.to_undirected(), random_state=42)`*

*After detecting communities, iterate over them and check for external edges: `[(u,v) for u,v in R.edges() if (u in members) != (v in members)]`*

In [ ]:
# Challenge 2 — your code here

# Step 1: run Louvain community detection
partition  = community_louvain.best_partition(R.to_undirected(), random_state=42)
comm_sizes = Counter(partition.values())
print(f'Communities detected: {len(comm_sizes)}')

# Step 2: find communities with zero external edges
# ...


In [ ]:
# Challenge 2 — visualisation (bonus +5 pts)
# Colour the referral network by community; highlight the suspicious group
# Hint: nx.spring_layout(R.to_undirected(), seed=42)

# ...

---
## 🔍 Challenge 3 — One Person, Two Places, Same Day (20 pts)

**Background:** Each claim has a `provider_id`, and each provider has a `city`. It is physically impossible for a patient to receive services in two distant cities on the same calendar day.

**Your task:** Find patient(s) with claims at geographically separated cities on the same day.

**Graph angle:** Enrich the patient→provider edges with location data. Group by `(patient_id, date_of_service)` and count distinct provider cities.

*Hint: `claims.merge(providers[['provider_id','city']], on='provider_id')` then `.groupby(['patient_id','date_of_service'])['city'].nunique()`*

In [ ]:
# Challenge 3 — your code here

# Step 1: enrich claims with provider city
c_city = claims.merge(providers[['provider_id', 'city']], on='provider_id')

# Step 2: count distinct cities per (patient, date)
# ...


In [ ]:
# Challenge 3 — visualisation (bonus +5 pts)
# Timeline plot for the suspicious patient
# Each city gets its own y-position; impossible-travel days get a vertical marker

# ...

---
## 🔍 Challenge 4 — Complexity Inflation (20 pts)

**Background:** CPT codes 99213→99215 represent increasing visit complexity. Billing 99215 (~$320, *high complexity*) is appropriate for roughly **12%** of established-patient visits across the industry. A provider billing it for the vast majority of visits is likely upcoding.

**Your task:** Find the provider whose 99215 rate is a statistical outlier.

**Graph angle:** Build a provider→procedure_code bipartite graph with edge weights = claim count. Compare each provider's weight distribution to the population using a Z-score.

*Hint: `z = (rate - mean) / std` — anything above 3σ warrants investigation.*

In [ ]:
# Challenge 4 — your code here

office_codes  = ['99213', '99214', '99215', '99203', '99204']
office_claims = claims[claims['procedure_code'].isin(office_codes)]

# Compute procedure code frequency distribution per provider
dist = (office_claims.groupby(['provider_id', 'procedure_code'])
                     .size().reset_index(name='n'))
dist['pct'] = dist.groupby('provider_id')['n'].transform(lambda x: x / x.sum())

# Focus on 99215 and compute Z-scores
# ...


In [ ]:
# Challenge 4 — visualisation (bonus +5 pts)
# Bar chart: 99215 rate for every provider
# Add horizontal lines for the population mean and the 3σ threshold

# ...

---
## 🔍 Challenge 5 — Pay Me Twice (20 pts)

**Background:** A duplicate claim is one with the same `patient_id`, `provider_id`, `procedure_code`, and `date_of_service` appearing more than once. The fraudster adds slight amount jitter to avoid naive exact-match detection.

**Your task:** Find the provider submitting systematically duplicate claims.

**Graph angle:** In a MultiGraph, duplicate claims appear as *parallel edges* between the same patient–provider pair, sharing matching `date` and `procedure` attributes.

*Hint: `claims.groupby([...]).agg(n_copies=('claim_id','count')).query('n_copies > 1')`*

In [ ]:
# Challenge 5 — your code here

key = ['patient_id', 'provider_id', 'procedure_code', 'date_of_service']

# Group by the natural claim key and find groups with >1 submission
# ...


In [ ]:
# Challenge 5 — visualisation (bonus +5 pts)
# MultiGraph subgraph for the suspicious provider
# Count edges between each (patient, provider) pair and highlight the parallel ones

# ...

---
## 🏆 Submit Your Findings

Fill in the `findings` dictionary and call `score_team('Your Team Name')`.

Entity IDs look like `'PRV-047'` (provider) or `'PAT-0089'` (patient).
For the referral ring, provide a list: `['PRV-031', 'PRV-032', 'PRV-033']`.

In [ ]:
# ── Fill in your answers ──────────────────────────────────────────────────────
findings = {
    'ghost_billing_provider':     None,   # e.g. 'PRV-047'
    'referral_ring_providers':    None,   # e.g. ['PRV-031', 'PRV-032', 'PRV-033']
    'impossible_travel_patient':  None,   # e.g. 'PAT-0089'
    'upcoding_provider':          None,   # e.g. 'PRV-022'
    'duplicate_billing_provider': None,   # e.g. 'PRV-055'
}
# ─────────────────────────────────────────────────────────────────────────────

_ANSWERS = {
    'ghost_billing_provider':     'PRV-047',
    'referral_ring_providers':    {'PRV-031', 'PRV-032', 'PRV-033'},
    'impossible_travel_patient':  'PAT-0089',
    'upcoding_provider':          'PRV-022',
    'duplicate_billing_provider': 'PRV-055',
}

def score_team(team_name: str = 'Your Team'):
    score   = 0
    results = []
    checks  = [
        ('ghost_billing_provider',     'Ghost Billing'),
        ('referral_ring_providers',    'Referral Ring'),
        ('impossible_travel_patient',  'Impossible Travel'),
        ('upcoding_provider',          'Upcoding'),
        ('duplicate_billing_provider', 'Duplicate Billing'),
    ]
    for key, label in checks:
        answer  = findings.get(key)
        correct = _ANSWERS[key]
        if answer is None:
            results.append((label, '—', '❌ Not answered', 0));  continue
        if isinstance(correct, set):
            got = set(answer) if isinstance(answer, (list, set)) else {answer}
            if got == correct:
                results.append((label, str(answer), '✅ Correct!', 20));  score += 20
            elif got & correct:
                n   = len(got & correct)
                pts = round(20 * n / len(correct))
                results.append((label, str(answer), f'⚠️  Partial ({n}/{len(correct)})', pts));  score += pts
            else:
                results.append((label, str(answer), '❌ Wrong', 0))
        else:
            if str(answer).strip() == str(correct).strip():
                results.append((label, answer, '✅ Correct!', 20));  score += 20
            else:
                results.append((label, answer, '❌ Wrong', 0))

    sep = '─' * 62
    print(f'\n{sep}')
    print(f'  SCORE REPORT  — {team_name}')
    print(sep)
    for label, given, verdict, pts in results:
        print(f'  {verdict:<25}  {label:<22}  +{pts:>2} pts   (you said: {given})')
    print(sep)
    print(f'  TOTAL: {score} / 100 pts')
    print(f'{sep}\n')
    return score

# Uncomment when ready:
# score_team('Team Awesome')

---
*Synthetic dataset — all names, IDs, and amounts are fictional and generated for this workshop.*